# 17-620 Lab 3 — Can Quantum Similarity Become a Classifier?

**Prepared by Daniel Justice · Carnegie Mellon University · 17-620 Quantum Machine Learning · Fall 2026**  
© 2026 Daniel Justice for original contributions; third-party materials retain their respective rights. Use and reuse are subject to applicable university policies.

**80 minutes | PennyLane 0.45.1 | Guided practice | Ungraded — no submission required**

### Our question
**We can encode classical data as quantum states and compare those states. Can those comparisons help us predict labels?**

In Lab 2 you measured squared overlaps. Today we will use them as a kernel for a classical support-vector machine (SVM), explore different feature maps, then investigate finite-shot uncertainty. A quantum feature map is **fixed**, not trained, in this lab; the **classical SVM** learns the separating rule.

**A note about Homework 1:** This lab uses a one-qubit, statevector-based demonstration on concentric circles. Homework 1 independently asks you to build two-qubit overlap circuits and kernel matrices on a *different dataset*. The lab demonstrates the idea; it does not complete the homework for you.

**No hardware account, cloud credentials, or assignment submission required.** Run cells in order. The exploratory changes are for your learning, not for grading.

## Architecture map — what is quantum, what is classical, and what is trained?

**Keep this map in view throughout the lab.** The input has **two classical features**, but the circuit has just **one qubit**. An encoding rule converts the two features into two rotation angles.

```text
                              FIXED / NOT TRAINED                      LEARNED
 CLASSICAL DATA                  QUANTUM CIRCUIT (SIMULATED)           CLASSICAL SVM
  x = (x₁, x₂)  --> θ(x), φ(x) --> |0>--RY(θ)--RZ(φ)-->|φ(x)>
                                    |                       |
                                    +-- states for all x ---+
                                                          |
                                        CLASSICAL INNER PRODUCTS
                                          K[i,j] = |<φ(i)|φ(j)>|²
                                                          |
                                     K_train (36 × 36) + labels (36)
                                                          |
                                         SVC(kernel="precomputed")
                                                          |
                                 learns support vectors, coefficients, bias
                                                          |
                       new x --> K_test row (1 × 36) --> predicted class
```


**The blue-sky quantum part:** \`RY\` and \`RZ\` are actual quantum circuit gates in PennyLane's circuit model, but \`default.qubit\` **simulates** them on a classical CPU. This notebook retrieves the exact statevectors with \`qml.state()\` and calculates their squared overlaps **classically**, rather than measuring every overlap with a physical QPU.

**The trained part:** \`SVC(kernel="precomputed", C=2.0).fit(K_train, train_labels)\` optimizes a **classical SVM**. The kernel values, feature-map type, and chosen \`strength\` are **not** adjusted by SVM training.

**Terminology bridge:** *Fidelity* in this lab is the *squared overlap* from Lab 2, for pure states. Only the computational route to the number differs.

## A — From fidelity to a learning problem (10 minutes)

In Lab 2, for quantum states prepared by an encoding circuit, we used the **fidelity kernel**

$$K(x,z)=|\langle\phi(x)|\phi(z)\rangle|^2.$$

Its value is between 0 and 1 for normalized pure states. Identical states have fidelity 1; orthogonal states have fidelity 0. It is not a distance, and a high value does not necessarily mean identical **classical inputs**.

We will consider a binary-label dataset: inner-ring vs outer-ring points. Labels are used to **train the SVM**, but **not to construct the fixed feature map**.

**Predict:** Could a straight line separate the two rings? What kind of transformation might make the labels easier to distinguish?

In [ ]:
import numpy as np
import pennylane as qml
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=3, suppress=True)
print("PennyLane version:", qml.__version__)

points, labels = make_circles(n_samples=48, factor=0.43,
                              noise=0.09, random_state=19)
train_points, test_points, train_labels, test_labels = train_test_split(
    points, labels, test_size=0.25, stratify=labels, random_state=17)
assert train_points.shape == (36, 2) and test_points.shape == (12, 2)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(train_points[:, 0], train_points[:, 1], c=train_labels,
           cmap="coolwarm", vmin=0, vmax=1, edgecolors="black", s=55, label="Train")
ax.scatter(test_points[:, 0], test_points[:, 1], c=test_labels,
           cmap="coolwarm", vmin=0, vmax=1, marker="s",
           edgecolors="black", s=70, label="Holdout")
ax.set(xlabel="Feature 1", ylabel="Feature 2", title="Two concentric classes")
ax.set_aspect("equal"); ax.legend()
plt.show()

## B — Change the feature map, change the geometry (15 minutes)

We use **one qubit** to keep every circuit visible. For input $x=(x_1,x_2)$ and a strength $s$, consider two predefined embeddings:

- **Coordinates:** $R_Y(s x_1)$, followed by $R_Z(s x_2)$.
- **Radius-aware:** $R_Y(s(x_1^2+x_2^2))$, followed by $R_Z(0.7 s x_2)$.

The radius-aware choice deliberately injects the prior knowledge that **distance from the origin** may matter. That is a classical engineering decision, **not a discovery made by a quantum computer**.

**Important:** Quantum operations are applied in notebook order: Y rotation first, then Z rotation. The global phase of a state does not affect fidelity.

To emphasize the idea rather than duplicate Homework 1's circuit-construction exercises, this lab asks PennyLane for statevectors, and takes their inner products on the classical computer. It is a **statevector simulation of a quantum feature map**, not a claimed hardware-speedup implementation. **The fidelity values are mathematically the same squared overlaps from Lab 2.** Only the calculation method differs: here we multiply simulated statevectors; Lab 2 used an inverse preparation circuit and the all-zero measurement probability.

In [ ]:
state_device = qml.device("default.qubit", wires=1)

@qml.qnode(state_device)
def embedded_state(x, style="radius", strength=1.7):
    if style == "coordinates":
        theta, phi = strength * x[0], strength * x[1]
    elif style == "radius":
        theta = strength * (x[0] ** 2 + x[1] ** 2)
        phi = 0.7 * strength * x[1]
    else:
        raise ValueError("style must be 'coordinates' or 'radius'")
    qml.RY(theta, wires=0)
    qml.RZ(phi, wires=0)
    return qml.state()

print("One data-dependent circuit:\n")
print(qml.draw(embedded_state)(train_points[0], "radius", 1.7))

def state_rows(samples, style="radius", strength=1.7):
    return np.asarray([np.asarray(embedded_state(x, style, strength))
                       for x in np.asarray(samples)], dtype=complex)

def fidelity_table(left_states, right_states):
    # Each row is a statevector. Conjugate the left side of the inner product.
    # SAME fidelity (squared overlap) as Lab 2, calculated directly from simulated statevectors.
    # Different implementation: Homework 1 estimates this probability with inverse circuits.
    return np.clip(np.abs(left_states.conj() @ right_states.T) ** 2, 0.0, 1.0)

small_states = state_rows(train_points[:6])
small_fidelity = fidelity_table(small_states, small_states)
assert small_fidelity.shape == (6, 6)
assert np.allclose(np.diag(small_fidelity), 1, atol=1e-10)
assert np.allclose(small_fidelity, small_fidelity.T, atol=1e-10)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(small_fidelity, cmap="viridis", vmin=0, vmax=1)
ax.set(xticks=range(6), yticks=range(6), xlabel="Point index",
       ylabel="Point index", title="Fidelity: first six training states")
fig.colorbar(im, ax=ax, label="Squared overlap")
plt.show()

### Circuit anatomy — what would actually be sent to a quantum device?

The state-preparation feature map is a **single-qubit circuit**. There is no entangling gate, measurement-based optimization, or trainable variational layer:

```text
Coordinates encoding:     |0> --- RY(s·x₁)  --- RZ(s·x₂) --------> |φ(x)>
Radius-aware encoding:    |0> --- RY(s·(x₁²+x₂²)) --- RZ(0.7s·x₂) -> |φ(x)>
```


For our default \`strength = 1.7\`, the \`radius-aware\` feature map uses **classically calculated** angles. The same two gates run for every input, just with different numerical angles.

**Important distinction:** \`qml.state()\` returns an ideal simulated statevector; it is **not** something a quantum device can directly read out in one shot. To estimate the same fidelity on quantum hardware, we could prepare \`|φ(x)>\`, **undo** the preparation of \`|φ(z)>\`, and measure the probability of zero:

```text
|0> -- RY(θx) -- RZ(φx) -- RZ(-φz) -- RY(-θz) -- measure Z
                                                 P(0) = |<φ(z)|φ(x)>|²
```


Below we draw **both circuits** and check that the exact simulated \`P(0)\` of the second one agrees with our classically computed statevector fidelity. This is an **illustration**, not the large overlap-circuit implementation required independently in Homework 1.

In [ ]:
# Side-by-side circuit diagrams for one fixed pair of inputs.
# We use the SAME angle rules as embedded_state, but expose the gates as an
# ordinary reusable quantum operation for an inverse-comparison demonstration.
def apply_feature_map(x, style="radius", strength=1.7):
    if style == "coordinates":
        theta = strength * x[0]
        phi = strength * x[1]
    elif style == "radius":
        theta = strength * (x[0] ** 2 + x[1] ** 2)
        phi = 0.7 * strength * x[1]
    else:
        raise ValueError("style must be 'coordinates' or 'radius'")
    qml.RY(theta, wires=0)
    qml.RZ(phi, wires=0)

overlap_device = qml.device("default.qubit", wires=1)

@qml.qnode(overlap_device)
def overlap_measurement_probabilities(x, z, style="radius", strength=1.7):
    apply_feature_map(x, style, strength)
    qml.adjoint(apply_feature_map)(z, style, strength)
    return qml.probs(wires=[0])

x_example, z_example = train_points[0], train_points[1]
print("STATE PREPARATION (fixed feature map):")
print(qml.draw(embedded_state)(x_example, "radius", 1.7))
print("\nOVERLAP MEASUREMENT CIRCUIT (equivalent to Lab 2):")
print(qml.draw(overlap_measurement_probabilities)(
    x_example, z_example, "radius", 1.7))

p_zero = float(overlap_measurement_probabilities(x_example, z_example)[0])
from_statevectors = float(small_fidelity[0, 1])
print(f"Probability of measuring 0: {p_zero:.6f}")
print(f"Statevector squared overlap: {from_statevectors:.6f}")
assert np.isclose(p_zero, from_statevectors, atol=1e-10)

# Optional instructor demo: compare the numerical rotation values and drawings
# when changing from radius-aware to coordinate encoding.


### Exercise B — Read the heatmap

**Before changing code:** Why is every diagonal entry 1? Is the entry at row 0, column 3 necessarily high because both points may have a label in common?

**TODO (optional short code experiment):** Change `selected_pair` to compare two other points. Predict whether their fidelity will be high before running. This cell works without edits.

In [ ]:
selected_pair = (0, 3)  # TODO: try another pair from 0 through 5
i, j = selected_pair
assert 0 <= i < 6 and 0 <= j < 6
print(f"Input x[{i}] = {train_points[i].round(3)}; label = {train_labels[i]}")
print(f"Input x[{j}] = {train_points[j].round(3)}; label = {train_labels[j]}")
print(f"Quantum-state fidelity = {small_fidelity[i, j]:.3f}")

## C — From pairwise similarities to predictions (20 minutes)

A **Gram matrix** contains one kernel entry for every pair of training points. For our pure-state fidelity kernel, the *exact* Gram matrix is symmetric and positive semidefinite (PSD), up to floating-point precision.

To predict a new point, the SVM needs its similarities to the **training points**, so the rectangular holdout matrix has shape **12 by 36**, not 12 by 12.

The SVM is a *classical* optimizer. Our quantum circuit does not learn its parameters. We use the same training and test split for all comparisons.

The code below provides an intact worked example so the lab remains runnable. Homework 1 requires you to implement a different kernel pipeline independently.

In [ ]:
def fit_quantum_view(style="radius", strength=1.7):
    reference_states = state_rows(train_points, style, strength)
    K_train = fidelity_table(reference_states, reference_states)
    K_test = fidelity_table(state_rows(test_points, style, strength),
                            reference_states)
    model = SVC(kernel="precomputed", C=2.0).fit(K_train, train_labels)
    predictions = model.predict(K_test)
    accuracy = accuracy_score(test_labels, predictions)
    return model, reference_states, K_train, K_test, accuracy

q_model, q_reference, K_train, K_test, q_accuracy = fit_quantum_view()
assert K_train.shape == (36, 36) and K_test.shape == (12, 36)
assert np.allclose(K_train, K_train.T, atol=1e-10)
assert np.allclose(np.diag(K_train), 1, atol=1e-10)
assert np.linalg.eigvalsh((K_train + K_train.T) / 2).min() > -1e-8
assert K_test.min() >= -1e-12 and K_test.max() <= 1 + 1e-12

print("Exact Gram matrix:", K_train.shape, "| Holdout matrix:", K_test.shape)
print(f"Smallest training Gram eigenvalue: {np.linalg.eigvalsh(K_train).min():.2e}")
print(f"Quantum-kernel SVM holdout accuracy: {q_accuracy:.3f}")
fig, ax = plt.subplots(figsize=(5.2, 4.1))
im = ax.imshow(K_train, vmin=0, vmax=1, cmap="viridis", aspect="auto")
ax.set(xlabel="Training example", ylabel="Training example",
       title="Exact Gram matrix: radius-aware feature map")
fig.colorbar(im, ax=ax, label="Fidelity")
plt.show()

### What did \`fit\` actually learn?

The SVM does **not** update \`strength\` or either rotation gate. Its learned decision rule is

$$
f(x)=\sum_{i\in\mathrm{support}} \alpha_i y_i\,K(x_i,x)+b.
$$

In scikit-learn, \`dual_coef_\` already contains the **signed** products \`αᵢ yᵢ\`, \`support_\` tells you which training examples are support vectors, and \`intercept_\` contains the bias. A positive decision score predicts \`classes_[1]\` for this binary classifier.

**Try this:** Print the learned support-vector indices and the number of support vectors, then compare the manual decision score of one test point against the SVM's own score. The code below makes the connection explicit without implementing the general Homework 1 classifier.

In [ ]:
# Inspect *classical* quantities produced by SVM training.
print("SVM class labels:", q_model.classes_)
print("Indices of support vectors:", q_model.support_)
print("Number of support vectors:", len(q_model.support_))
print("Dual coefficients (first five):", q_model.dual_coef_[0, :5])
print("Learned intercept:", q_model.intercept_[0])
print("Feature-map strength (still fixed):", 1.7)

test_index = 0
# The model stores signed dual coefficients and a bias. For a new input,
# it needs similarities to support vectors, not the raw statevector itself.
manual_score = (
    float(q_model.dual_coef_[0] @ K_test[test_index, q_model.support_])
    + float(q_model.intercept_[0])
)
library_score = float(q_model.decision_function(K_test[[test_index]])[0])
predicted = int(q_model.predict(K_test[[test_index]])[0])
print(f"New sample {test_index}: score={manual_score:.5f}, "
      f"predicted class={predicted}, actual class={test_labels[test_index]}")
assert np.isclose(manual_score, library_score, atol=1e-8)


### Classical comparison

A fair comparison needs **strong classical baselines**. We include both a linear SVM and an RBF-kernel SVM using exactly the same inputs and held-out examples.

The engineered radius feature may work very well on this toy dataset. So may the classical RBF kernel. **High accuracy alone does not establish a quantum advantage.**

In [ ]:
linear_model = SVC(kernel="linear", C=2.0).fit(train_points, train_labels)
rbf_model = SVC(kernel="rbf", C=2.0, gamma=1.0).fit(train_points, train_labels)
scores = {
    "Linear SVM": accuracy_score(test_labels, linear_model.predict(test_points)),
    "RBF SVM": accuracy_score(test_labels, rbf_model.predict(test_points)),
    "Quantum-fidelity SVM": q_accuracy,
}
for name, score in scores.items():
    print(f"{name:24s} holdout accuracy = {score:.3f}")

def decision_plot(style="radius", strength=1.7, show_all=True):
    quantum, states, Ktr, Kte, heldout = fit_quantum_view(style, strength)
    xs = np.linspace(-1.3, 1.3, 35)
    gx, gy = np.meshgrid(xs, xs)
    grid = np.column_stack([gx.ravel(), gy.ravel()])
    grid_fidelity = fidelity_table(state_rows(grid, style, strength), states)
    boundaries = [linear_model.predict(grid).reshape(gx.shape),
                  rbf_model.predict(grid).reshape(gx.shape),
                  quantum.predict(grid_fidelity).reshape(gx.shape)]
    titles = ["Classical linear", "Classical RBF",
              f"Quantum: {style}, s={strength:g}"]
    fig, axes = plt.subplots(1, 3, figsize=(12.2, 3.8), sharex=True, sharey=True)
    for ax, boundary, title in zip(axes, boundaries, titles):
        ax.contourf(gx, gy, boundary, levels=[-0.5, 0.5, 1.5],
                    cmap="coolwarm", alpha=0.28)
        ax.scatter(train_points[:, 0], train_points[:, 1],
                   c=train_labels, vmin=0, vmax=1, cmap="coolwarm",
                   s=28, edgecolors="black", linewidths=0.4)
        ax.scatter(test_points[:, 0], test_points[:, 1],
                   c=test_labels, vmin=0, vmax=1, cmap="coolwarm",
                   s=55, marker="s", edgecolors="black", linewidths=0.8)
        ax.set(title=title, xlabel="Feature 1", xlim=(-1.3, 1.3),
               ylim=(-1.3, 1.3))
        ax.set_aspect("equal")
    axes[0].set_ylabel("Feature 2")
    fig.suptitle("Shading = prediction; circle = train; square = holdout", fontsize=11)
    fig.tight_layout()
    plt.show()
    return heldout

demo_accuracy = decision_plot("radius", 1.7)
assert 0 <= demo_accuracy <= 1

## D — Inquiry: can a different feature map change the classifier? (20 minutes)

An SVM's predictions depend on the notion of similarity provided to it. **Change the encoding, not the labels**, and ask whether the trained boundary changes.

1. **Predict first:** Would the coordinate or radius-aware feature map work better for these concentric circles?
2. Run the next cell. Inspect **training vs holdout** scores rather than deciding from a single picture.
3. Change just one setting. Does very large `strength` help or can rotation periodicity produce aliasing?
4. Compare your observations with the RBF baseline. What knowledge did we hand-design into the radius-aware embedding?

**TODO:** Modify a value or add your own `(style, strength)` setting. There is intentionally no required answer or submission.

In [ ]:
settings_to_try = [
    ("coordinates", 2.5),
    ("radius", 1.7),
    ("radius", 4.5),
]  # TODO: try an additional pair, or change one strength.

for style, strength in settings_to_try:
    model, states, matrix, holdout, accuracy = fit_quantum_view(style, strength)
    training_accuracy = model.score(matrix, train_labels)
    print(f"{style:12s} strength={strength:3.1f}  "
          f"train={training_accuracy:.3f}  holdout={accuracy:.3f}")

### Optional deeper inquiry — Are same-label points always similar?

A kernel is built **without labels**. After training, we can use the training labels **only as a diagnostic** to ask whether the feature map tends to place same-class points closer together.

The code below compares the **off-diagonal mean fidelity within classes** with the **mean fidelity between classes**. Higher within-class similarity is a useful clue for these data, but does **not** guarantee high generalization.

**TODO:** Predict which embedding will have the larger gap, run the cell, and explain *why*. Do **not** use the test labels to pick a feature map. This is optional if time is short.

In [ ]:
def class_alignment_stats(matrix, y):
    same = y[:, None] == y[None, :]
    off_diagonal = ~np.eye(len(y), dtype=bool)
    within = matrix[same & off_diagonal]
    between = matrix[(~same) & off_diagonal]
    return float(within.mean()), float(between.mean())

for style, strength in [("coordinates", 2.5), ("radius", 1.7)]:
    _, _, gram, _, _ = fit_quantum_view(style, strength)
    within, between = class_alignment_stats(gram, train_labels)
    print(f"{style:12s} within={within:.3f}  between={between:.3f}  "
          f"gap={within - between:.3f}")

# Challenge: find two *different-label* points that nevertheless have
# unusually high fidelity. Explain why labels do not determine the kernel.
different_class = train_labels[:, None] != train_labels[None, :]
u, v = np.unravel_index(np.argmax(np.where(different_class, K_train, -1.0)),
                       K_train.shape)
print(f"Different-label counterexample: pair ({u}, {v}), fidelity {K_train[u, v]:.3f}")
assert train_labels[u] != train_labels[v]

### Optional live explorer

The controls let you redraw the quantum decision region while holding the dataset and classical baselines fixed. Move the slider **after it finishes computing**, not continuously. The default static plot above remains usable if widgets are unavailable in your notebook interface.

In [ ]:
def explore_setting(style="radius", strength=1.7):
    result = decision_plot(style, strength)
    print(f"Quantum holdout accuracy: {result:.3f}")

style_picker = widgets.Dropdown(options=[("Radius-aware", "radius"),
                                          ("Coordinate rotations", "coordinates")],
                                value="radius", description="Encoding:")
scale_picker = widgets.FloatSlider(value=1.7, min=0.5, max=4.5,
                                   step=0.4, description="Strength:",
                                   continuous_update=False, readout_format=".1f")
controls = widgets.HBox([style_picker, scale_picker])
output = widgets.interactive_output(explore_setting,
                                    {"style": style_picker, "strength": scale_picker})
display(controls, output)

## E — From exact probabilities to finite shots (10 minutes)

So far we used an exact statevector simulator. On hardware, a standard overlap-test circuit would return measurement outcomes after repeated **shots**. If a particular "all-zero" outcome has probability $p=K(x,z)$, the observed success fraction has

$$\mathbb{E}[\hat p]=p,\qquad \operatorname{SD}(\hat p)=\sqrt{\frac{p(1-p)}{N}}.$$

To focus on the statistics rather than repeat Homework 1's circuit implementation, we simulate these outcomes using binomial draws **from our exact fidelity**. This is *ideal shot sampling*, **not a hardware run** and **not a gate-noise model**. Larger $N$ reduces typical uncertainty, but no single experiment is guaranteed to move closer to $p$.

In [ ]:
upper_rows, upper_cols = np.triu_indices(len(train_points), k=1)
best = np.argmin(np.abs(K_train[upper_rows, upper_cols] - 0.5))
a, b = int(upper_rows[best]), int(upper_cols[best])
p_exact = float(K_train[a, b])
print(f"Pair ({a}, {b}): exact fidelity = {p_exact:.3f}")

rng = np.random.default_rng(620)
shot_counts = [64, 256, 2048]
for n in shot_counts:
    estimate = rng.binomial(n, p_exact) / n
    predicted_sd = np.sqrt(p_exact * (1 - p_exact) / n)
    print(f"{n:4d} shots: estimate {estimate:.3f}; expected SD {predicted_sd:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharex=True)
for ax, n in zip(axes, [64, 256]):
    repeated_estimates = rng.binomial(n, p_exact, size=350) / n
    ax.hist(repeated_estimates, bins=19, range=(0, 1), color="slateblue", alpha=0.8)
    ax.axvline(p_exact, color="black", linestyle="--", label="Exact fidelity")
    ax.set(title=f"{n} shots, 350 hypothetical repeats", xlabel="Estimated fidelity",
           ylabel="Count", xlim=(0, 1))
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### Exercise E — Scientific interpretation

**Predict:** If we repeated an experiment with 256 shots and then with 2048 shots, must the second estimate be closer to the truth? Why not?

**Think ahead:** What happens if the same finite-shot procedure independently estimates *all* entries of a Gram matrix? Unlike the exact pure-state Gram matrix, an independently estimated noisy matrix need not be symmetric or PSD unless we impose additional structure. What could an SVM do with such a matrix?

**Hardware extension (not required):** Real devices add state-preparation, gate, measurement, compilation, queue, and connectivity effects. Those are separate from ideal binomial fluctuations. We will address them in later hardware/noise activities.

## F — Exit reflection and bridge to Lab 4 (5 minutes)

Discuss with a classmate, or jot brief notes for yourself (**no submission required**):

1. What was encoded into the quantum state, and what was learned by the classical SVM?
2. Why might a radius-aware embedding perform well on concentric circles? Is this quantum advantage?
3. Why can a classical RBF kernel be a strong comparator?
4. How does finite-shot sampling change our confidence in the entries of a kernel matrix?
5. **Next question:** If the fixed feature map is a poor choice, how might a *trainable*, parameterized quantum circuit improve the model? What difficulties might gradients, optimization, and noise introduce?

**Takeaway:** Encoding selects a geometry; a kernel turns it into similarities; a classical learner makes predictions. The interesting science is in the feature map, the comparison to classical methods, and the practical costs—not in declaring a quantum winner.